# 05 — Retrieval Demo (Full): Hybrid Search, Rerank, HyDE, Breadth Routes & the Router

This notebook demonstrates the **complete retrieval pipeline** of the offer assistant,
ported 1:1 from the app (`app/src/rag_system/`). It builds on NB4 (hybrid search)
and adds:

1. **LLM reranking** — score hybrid candidates, keep top-3, refuse if best < 5/10
2. **HyDE** — hypothetical document as a third RRF arm (vector + BM25 + HyDE)
3. **Breadth routes** — statistics (full scan + code reduce), comparison (map-reduce),
   year list (deterministic metadata scan)
4. **The router** — ID-aware retrieval, deterministic price, refusal, clarification,
   aggregation, compound RAG, and the breadth routes

```
Question
  │
  ├─ contains AG####?
  │      └─► ID-aware retrieval (metadata filter, NO fallback)
  │          ├─ price question?  → DETERMINISTIC price from metadata (no LLM)
  │          ├─ not found?       → REFUSAL ("AG9999 wurde nicht gefunden")
  │          └─ else             → rerank + gate → cited RAG answer
  │
  ├─ statistics (wie viele / durchschnitt / ausreißer)
  │      └─► FULL SCAN: LLM reads EVERY offer (map) → deterministic CODE
  │          computes count/mean/range/outliers (reduce). No retrieval, no top-k.
  │
  ├─ comparison (vergleiche / unterschiede)
  │      └─► MAP-REDUCE: topic retrieval (top-20, dedup by offer)
  │          → LLM map: one fact line per offer → LLM reduce: comparison.
  │          (= LlamaIndex `tree_summarize`, hand-rolled to keep our
  │            hybrid retrieval pipeline)
  │
  ├─ year list (year + "angebot(e)")
  │      └─► DETERMINISTIC METADATA SCAN: every offer's `datum` is checked —
  │          complete list, no LLM, no retrieval.
  │
  ├─ ambiguous (price/date/term, no offer ref, no year)
  │      └─► CLARIFICATION: "Meinst du eines dieser? AG#### (datum, preis) · …"
  │
  ├─ aggregation (welche angebote / alle / mehr als)
  │      └─► RAG + explicit limitation note (full SQL path is the next step)
  │
  └─ default
         └─► grounded RAG (hybrid + optional HyDE arm → rerank → gate → answer)
```

Citations: inline `AG####` (no brackets); a verbatim quote directly followed by
its citation is upgraded deterministically → `„…" — 01.05.2026 · AG0085 (S. 4)`.

## Setup — Environment, LLM & Index

Same as NB4: load `.env`, OpenAI client (thinking OFF, temp 0),
Chroma collection, Ollama embedding model.

In [1]:
import os, re, json, time
from pathlib import Path
from dotenv import load_dotenv

# Load .env (walk up: notebooks/ -> 01-submission/ -> final-project/)
for path in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    dotenv_file = path / ".env"
    if dotenv_file.exists():
        load_dotenv(dotenv_file, override=True)
        print(f"✅ Loaded .env from {dotenv_file}")
        break

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "")
LLM_MODEL = os.getenv("LLM_MODEL", "")
LLM_API_KEY = os.getenv("LLM_API_KEY", "not-needed")
EMBED_BASE_URL = os.getenv("EMBED_BASE_URL", "")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")

DEMO_DIR = Path.cwd().parent            # 01-submission
CHROMA_DIR = DEMO_DIR / "data" / "db" / "chroma"
COLLECTION = "offers"

# Retrieval pipeline parameters (same defaults as the app's config.py)
RRF_K = 60
W_VEC, W_BM25 = 0.5, 0.5
RERANK_TOP_N = 10
KEEP = 3
REFUSAL_THRESHOLD = 5.0

# --- LLM (remote vLLM, OpenAI-compatible; thinking OFF, temperature 0) ---
from openai import OpenAI
llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

def llm_chat(prompt: str, system: str | None = None) -> str:
    """One LLM call. `system` (optional) is sent as a system message FIRST —
    the vLLM endpoint rejects requests where a system message is not at the
    beginning, so it is always prepended, never appended."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        temperature=0,
        max_tokens=4096,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return resp.choices[0].message.content

# --- Index (loaded from disk, exactly like the app) ---
import chromadb
from llama_index.embeddings.ollama import OllamaEmbedding
from types import SimpleNamespace

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_collection(COLLECTION)
embed_model = OllamaEmbedding(
    model_name=EMBED_MODEL,
    base_url=EMBED_BASE_URL.replace("/v1", ""),
)
print(f"✅ Index: {collection.count()} chunks in '{COLLECTION}' ({CHROMA_DIR})")
print(f"✅ LLM: {LLM_MODEL} at {LLM_BASE_URL} (thinking off, temp 0)")
print(f"✅ Pipeline: RRF k={RRF_K} w={W_VEC}/{W_BM25} → rerank top-{RERANK_TOP_N} → keep {KEEP} → refusal < {REFUSAL_THRESHOLD}")

✅ Loaded .env from rag-system-for-quote-history/.env
✅ Index: 10 chunks in 'offers' (rag-system-for-quote-history/data/db/chroma)
✅ LLM: qwen3.8:27b at https://vllm.srv-prod-7.<INTERNAL-HOST>/v1 (thinking off, temp 0)
✅ Pipeline: RRF k=60 w=0.5/0.5 → rerank top-10 → keep 3 → refusal < 5.0


## Step 1: Hybrid Search (BM25 + Vector + RRF, optional HyDE arm)

Same core as NB4, extended: `rrf_fuse` accepts a third list
(HyDE vector hits) with its own weight. `hybrid_search` takes an
optional `hyde_passage` parameter.

In [10]:
def vector_search(query, top_n=10, angebot_id=None):
    """Vector search via the same embedding model the index was built with."""
    q_emb = embed_model.get_text_embedding(query)
    where = {"angebot_id": angebot_id} if angebot_id else None
    res = collection.query(query_embeddings=[q_emb], n_results=top_n,
                           where=where,
                           include=["documents", "metadatas", "distances"])
    out = []
    for _id, _doc, _meta, _dist in zip(res["ids"][0], res["documents"][0],
                                       res["metadatas"][0], res["distances"][0]):
        meta = dict(_meta or {})
        meta["vec_score"] = max(0.0, 1.0 - _dist / 2.0)   # L2 -> similarity
        out.append(SimpleNamespace(node_id=_id, text=_doc, metadata=meta))
    return out

def bm25_search(query, top_n=10, angebot_id=None):
    """Keyword search over the indexed chunks (optionally restricted to one offer).

    Only chunks with a positive BM25 score are returned — same as the app's
    retriever. For a query that matches nothing (e.g. "AG9999") this yields
    an empty list, so the ID-aware route can answer "not found" instead of
    reranking junk candidates.
    """
    scores = bm25.get_scores(word_tokenize(query.lower()))
    if angebot_id:
        for i, n in enumerate(node_list):
            if n.metadata.get("angebot_id") != angebot_id:
                scores[i] = -1.0
    top_ids = scores.argsort()[::-1][:top_n]
    return [node_list[i] for i in top_ids if scores[i] > 0]

def rrf_fuse(vec_results, bm25_results, w_vec=W_VEC, w_bm25=W_BM25, k=RRF_K, top_n=10,
             hyde_results=None, w_hyde=None):
    """Merge ranked node lists via weighted Reciprocal Rank Fusion.

    Optionally fuses a third list (HyDE: vector hits for a hypothetical
    answer passage) with its own weight.
    """
    scores, node_by_id = {}, {}
    for rank, node in enumerate(vec_results):
        scores[node.node_id] = scores.get(node.node_id, 0.0) + w_vec / (k + rank + 1)
        node_by_id[node.node_id] = node
    for rank, node in enumerate(bm25_results):
        scores[node.node_id] = scores.get(node.node_id, 0.0) + w_bm25 / (k + rank + 1)
        node_by_id[node.node_id] = node
    if hyde_results is not None and w_hyde:
        for rank, node in enumerate(hyde_results):
            scores[node.node_id] = scores.get(node.node_id, 0.0) + w_hyde / (k + rank + 1)
            node_by_id[node.node_id] = node
    ranked = sorted(scores, key=scores.get, reverse=True)[:top_n]
    out = []
    for nid in ranked:
        node = node_by_id[nid]
        node.metadata["rrf_score"] = scores[nid]
        out.append(node)
    return out

def hybrid_search(query, top_n=RERANK_TOP_N, angebot_id=None, hyde_passage=None):
    """Vector + BM25, merged with RRF. Returns top_n candidates.

    If `angebot_id` is given (the question names a specific offer), both
    search paths are restricted to that offer — same as the app's query.py.
    If `hyde_passage` is given (HyDE), it is embedded and vector-searched as a
    third RRF arm with weights 0.4/0.3/0.3 (vs 0.5/0.5 without HyDE).
    """
    vec = vector_search(query, top_n=top_n, angebot_id=angebot_id)
    kw = bm25_search(query, top_n=top_n, angebot_id=angebot_id)
    if hyde_passage:
        hy = vector_search(hyde_passage, top_n=top_n, angebot_id=angebot_id)
        return rrf_fuse(vec, kw, w_vec=W_VEC_HYDE, w_bm25=W_BM25_HYDE,
                        w_hyde=W_HYDE, hyde_results=hy, top_n=top_n)
    return rrf_fuse(vec, kw, top_n=top_n)

print("✅ Hybrid search ready (vector + BM25 + RRF, weights 0.5/0.5; HyDE arm optional)")

✅ Hybrid search ready (vector + BM25 + RRF, weights 0.5/0.5; HyDE arm optional)


## Step 2: Rerank, Refusal Gate & Cited Answer

Same as NB4, plus `_pre_ranked` param so the router can pass its
already-gated candidates without re-reranking.

In [8]:
def _strip_think(text):
    """Remove a think block if the model thought despite thinking being off."""
    open_tag = "\u003cthink"
    close_tag = "\u003c/think\u003e"
    if open_tag in text:
        text = text.split(open_tag, 1)[1].strip()
    if close_tag in text:
        text = text.split(close_tag, 1)[0].strip()
    return text

PAGE_RE = re.compile(r"\[Seite (\d+) von \d+\]")

def _norm(s):
    """Normalize whitespace + quote variants for robust quote matching."""
    s = re.sub(r"\s+", " ", s)
    for a, b in [("\u201e", '"'), ("\u201c", '"'), ("\u201d", '"'),
                 ("\u201a", "'"), ("\u2018", "'"), ("\u2019", "'"),
                 ("\u2026", "...")]:
        s = s.replace(a, b)
    return s.strip()

def page_of_quote(chunk_text, quote):
    """Deterministic page lookup: find the verbatim quote in the chunk text and
    return the number of the LAST '[Seite X von Y]' marker before it.
    Returns None if the quote cannot be located (never invent a page)."""
    nq, nt = _norm(quote), _norm(chunk_text)
    pos = nt.find(nq)
    if pos < 0:
        return None  # quote not verbatim in chunk -> no page, never guess
    pages = [int(m.group(1)) for m in PAGE_RE.finditer(nt) if m.start() < pos]
    return pages[-1] if pages else 1

def _format_date(value):
    """ISO date (2026-05-01) -> German format (01.05.2026)."""
    if not value:
        return None
    parts = str(value).split("-")
    if len(parts) == 3 and all(p.isdigit() for p in parts):
        y, m, d = parts
        return f"{d}.{m}.{y}"
    return str(value)

def llm_rerank(query, candidates, keep=KEEP):
    """Score hybrid candidates with the LLM (0-10) and return the top `keep`."""
    # Chunks can be ~4k chars; key facts (e.g. payment terms) often sit far
    # into the text, so keep a generous window for the rerank model.
    numbered = "\n\n".join(f"[{i}] {n.text[:2000]}"
                           for i, n in enumerate(candidates, 1))
    prompt = (
        "Given the query, score each passage 0-10 for how well it ANSWERS the "
        "query. If the query has several parts, a passage that fully answers "
        "ANY one part deserves a high score. "
        "Return ONLY a JSON object mapping passage number to score.\n\n"
        f"Query: {query}\n\nPassages:\n{numbered}"
    )
    # Retry loop: the remote LLM occasionally returns an empty/malformed
    # completion. Retry up to 3x with a short backoff.
    t0 = time.time()
    scores = {}
    for attempt in range(3):
        text = llm_chat(prompt,
                        system="You are a precise ranking engine. Output JSON only.")
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            try:
                scores = {int(k): float(v)
                          for k, v in json.loads(m.group(0)).items()}
                break
            except (json.JSONDecodeError, ValueError, TypeError):
                scores = {}
        if attempt < 2:
            time.sleep(0.5 * (attempt + 1))
    elapsed = time.time() - t0
    if not scores:
        # Rerank failed — fall back to retrieval order.
        print(f"   rerank: {elapsed:.1f}s — no scores, keeping retrieval order")
        return candidates[:keep]
    for i, node in enumerate(candidates, 1):
        node.metadata["rerank_score"] = scores.get(i, 0.0)
    ranked = sorted(candidates, key=lambda n: n.metadata["rerank_score"],
                    reverse=True)
    print(f"   rerank: {elapsed:.1f}s over {len(candidates)} candidates")
    return ranked[:keep]

AG_RE = re.compile(r"\bAG\d{4}\b")

SYSTEM_PROMPT = """Du bist der Angebot-Assistent eines Post-Production-Studios. Du beantwortest
Fragen zu früheren Angeboten des Studios.

REGELN:
1. Antworte NUR auf Basis des bereitgestellten Kontexts. Erfinde keine Zahlen,
   Konditionen oder Formulierungen.
2. Jede konkrete Aussage (Preis, Datum, Zahlungsbedingung, Formulierung) muss
   einem konkreten Angebot zugeordnet sein. Zitiere inline als AG#### —
   OHNE eckige Klammern, direkt nach der zugehörigen Aussage.
3. Wörtliche Zitate nur, wenn sie etwas beitragen: Bei exakten
   Formulierungen, Konditionen oder umstrittenen Aussagen führe die
   entscheidende Passage exakt so an, wie sie im Kontext steht, in
   deutschen Anführungszeichen („…"). Bei einfachen Fakten reicht die
   Paraphrase mit Citation — nicht jede Antwort braucht ein wörtliches
   Zitat. Das Citation-Schema steht NACH dem Zitat — nicht im Satz:
   "Wörtlich heißt es: „Zahlungsziel: 14 Tage ab Rechnungsdatum." AG0085"
   (Datum, Bindestrich und Seitenzahl ergänzt das Post-Processing automatisch).
4. Zitiere jede Quelle pro Aussage NUR EINMAL — nicht im Satz UND nochmal am
   Satzende, und nicht am Ende eines Satzes UND nochmal am Anfang des
   nächsten. Ein Satz, der bereits mit AG#### belegt ist, braucht keine
   zweite Quellenangabe.
5. Vermische KEINE Daten aus verschiedenen Angeboten zu einer Antwort. Wenn du
   über mehrere Angebote vergleichst, nenne für jeden Wert das Angebot, aus dem
   er stammt.
6. Wenn die Frage ein konkretes Angebot nennt (z.B. "AG0085") und dieses NICHT
   im Kontext ist: sage das klar ("AG0085 wurde nicht gefunden") und antworte
   nicht spekulativ.
7. Wenn der Kontext die Frage nicht beantwortet: lehne ab in einem Satz
   ("Das steht in den vorliegenden Angeboten nicht.") — keine Schätzung.
8. Wenn die Frage mehrdeutig ist (z.B. "Wie hoch war der Preis?" ohne
   Angebotsbezug): antworte NICHT. Stelle stattdessen eine kurze Rückfrage und
   liste die Kandidaten aus dem Kontext auf, z.B.:
   "Meinst du eines dieser Angebote? AG0085 (01.05.2026, 5.844,52 €) ·
   AG0086 (…, 8.160,80 €) · AG0090 (…, 1.251,03 €)"
9. Antworte auf Deutsch. Struktur: zuerst die direkte Antwort (1–2 Sätze),
   dann Details mit eingebundenen Zitaten, am Ende eine Quellenzeile als
   Übersicht: "Quellen: AG0085, AG0086" — kommagetrennt, OHNE Bindestriche,
   Sternchen oder Aufzählungszeichen.
10. Der Index enthält insgesamt {offer_count} Angebote. Dein Kontext zeigt nur
    die ähnlichsten Treffer — behaupte niemals, der Index enthalte nur die im
    Kontext sichtbaren Angebote.
11. Nenne Preise, Beträge oder Kosten NUR, wenn die Frage danach fragt.
    Der Gesamtbetrag eines Angebots ist kein Beleg für den Preis eines
    einzelnen Leistungspunkts — verwechsle beides nicht."""

_PRICE_WORDS = (
    "preis", "preise", "kosten", "kostet", "betrag", "beträgt", "betrage",
    "euro", "€", "summe", "honorar", "satz", "tatsachen", "rechnungsbetrag",
)

def question_asks_about_price(question):
    """True if the question is about prices, costs or amounts.

    The facts block only includes the total offer price in that case —
    otherwise the model is tempted to quote totals the user never asked
    for (and to confuse the offer total with a line-item price).
    """
    q = question.lower()
    return any(word in q for word in _PRICE_WORDS)

def _facts_block(ranked, question):
    """Structured per-offer facts for the answer prompt (from metadata,
    not from the model's reading of the text)."""
    facts, seen = [], set()
    include_price = question_asks_about_price(question)
    for node in ranked:
        oid = node.metadata.get("angebot_id")
        if oid in seen:
            continue
        seen.add(oid)
        line = f"- {oid}: datum={node.metadata.get('datum') or '—'}"
        if include_price:
            line += f", preis={format_price(node.metadata.get('preis')) or '—'}"
        facts.append(line)
    return "\n".join(facts) if facts else "(keine strukturierten Daten)"

def answer(query, top_n=RERANK_TOP_N, keep=KEEP, _pre_ranked=None):
    """Full grounded pipeline: hybrid -> rerank -> (refuse if low) -> cited answer.

    If the question names a specific offer (AG####), retrieval is restricted
    to that offer via a metadata filter — same as the app's query.py.
    Returns (answer_text, ranked_nodes, top_score).
    """
    m = AG_RE.search(query)
    angebot_id = m.group(0) if m else None
    if _pre_ranked is not None:
        # Router already ran hybrid search + rerank + gate.
        ranked = _pre_ranked
    else:
        ranked = llm_rerank(query, candidates=hybrid_search(query, top_n=top_n, angebot_id=angebot_id), keep=keep)
    top_score = ranked[0].metadata.get("rerank_score", 0.0) if ranked else 0.0

    if _pre_ranked is None and top_score < REFUSAL_THRESHOLD:
        msg = (f"Die Angebote enthalten dazu keine verlässliche Antwort "
               f"(bester Kandidat: {top_score:.0f}/10).")
        return msg, ranked, top_score

    facts_block = _facts_block(ranked, query)
    context = "\n\n".join(
        f"[{i}] ({n.metadata.get('angebot_id')}, "
        f"score {n.metadata.get('rerank_score', 0.0):.2f})\n{n.text}"
        for i, n in enumerate(ranked, 1)
    )
    system_prompt = SYSTEM_PROMPT.format(offer_count=_offer_count())
    user_prompt = (f"Strukturierte Daten der Treffer:\n{facts_block}\n\n"
                   f"Kontext:\n{context}\n\nFrage: {query}")
    answer_text = _strip_think(llm_chat(user_prompt, system=system_prompt)).strip()
    answer_text = upgrade_citations(answer_text, ranked)
    return answer_text, ranked, top_score

# --- Deterministic citation upgrade (same as the app's citation_markup.py) ---
# The model cites offers as plain AG#### (no brackets) and quotes the
# decisive passage verbatim in „…". A citation that DIRECTLY follows a
# verbatim quote is upgraded to the full schema:
#   „…" — 01.05.2026 · AG0085 (S. 4)
# The page comes from the "[Seite X von Y]" markers inside the chunk text,
# the date from the chunk metadata. A quote that cannot be located verbatim
# never gets a page (never guess). The trailing source list stays plain.

QUOTE_RE = re.compile(r"[\u201e\"](.+?)[\u201c\"]", flags=re.DOTALL)
CITE_RE = re.compile(
    r"(\[(AG\d{4})(?:\s*\|[^\]]*)?\]|(?<![A-Za-z0-9\[])(AG\d{4})(?!\d))"
    r"(?!\s*(?:,|\()\s*(?:S\.|Seite)\s*\d)"
)
_CITE_GAP_RE = re.compile(r"[A-Za-z]")
_SENTENCE_END_RE = re.compile(r"[.!?][\u201c\"]?\s+")

def _dedupe_sentence_start_citations(content):
    """Drop a citation that starts a sentence when the same offer was
    already cited in the previous sentence.

    The model sometimes emits the id both at the end of one sentence and
    again at the start of the next ("…möglich sind. AG0085 Wörtlich heißt
    es: „…" AG0085") — the sentence-start one is redundant. Only citations
    DIRECTLY after a sentence end (no letters in between) are removed, so
    "Quellen: AG0085, AG0086" and in-sentence citations are untouched.
    A citation that directly follows a verbatim quote is the real one and
    is never removed.
    """
    matches = list(CITE_RE.finditer(content))
    if len(matches) < 2:
        return content
    quote_ends = [m.end() for m in QUOTE_RE.finditer(content)]
    seen = set()
    removals = []
    for m in matches:
        offer_id = m.group(2) or m.group(3)
        # Never remove the citation that directly follows a quote.
        if any(not _CITE_GAP_RE.search(content[p:m.start()])
               for p in quote_ends if p < m.start()):
            seen.add(offer_id)
            continue
        gap = content[:m.start()]
        ends_sentence = False
        for sm in _SENTENCE_END_RE.finditer(gap):
            if not _CITE_GAP_RE.search(gap[sm.end():]):
                ends_sentence = True
        if ends_sentence and offer_id in seen:
            # Also swallow the trailing space so no double space remains.
            end = m.end()
            while end < len(content) and content[end] == " ":
                end += 1
            removals.append((m.start(), end))
        seen.add(offer_id)
    for start, end in reversed(removals):
        content = content[:start] + content[end:]
    return content

def upgrade_citations(answer_text, ranked):
    """Post-processing: upgrade a citation that directly follows a verbatim
    quote to the full schema: AG0085 -> 01.05.2026 · AG0085 (S. 4), with an
    em dash between quote and citation. Deterministic — no LLM involved.
    Citations that cannot be resolved (quote not located in the chunk) or
    that are not directly after a quote (e.g. the trailing source list)
    stay plain AG####."""
    content = _dedupe_sentence_start_citations(answer_text)
    quotes = [(m.end(), m.group(1).strip()) for m in QUOTE_RE.finditer(content)]
    if not quotes:
        return content
    # map offer id -> chunk texts / dates (only the kept, high-scoring nodes)
    texts, dates = {}, {}
    for node in ranked:
        oid = node.metadata.get("angebot_id")
        texts.setdefault(oid, []).append(node.text)
        dates.setdefault(oid, _format_date(node.metadata.get("datum")))
    pairs = []
    for m in CITE_RE.finditer(content):
        offer_id = m.group(2) or m.group(3)
        # Only upgrade when a verbatim quote ENDS directly before the
        # citation (gap without letters).
        candidates = [(p, q) for p, q in quotes
                      if p < m.start()
                      and not _CITE_GAP_RE.search(content[p:m.start()])]
        if not candidates:
            continue
        quote_pos, quote = candidates[-1]
        page = None
        for t in texts.get(offer_id, []):
            page = page_of_quote(t, quote)
            if page:
                break
        if not page:
            continue  # quote not located — never guess a page or date
        date = dates.get(offer_id)
        parts = [date] if date else []
        parts.append(f"{offer_id} (S. {page})" if page else offer_id)
        repl = " · ".join(parts)
        # The citation block is separated from the quote by an em dash.
        gap = content[quote_pos:m.start()]
        if "—" not in gap and "–" not in gap:
            repl = (" — " if not gap else "— ") + repl
        pairs.append((m, repl))
    # replace from the end so earlier positions stay valid
    for m, repl in reversed(pairs):
        content = content[:m.start()] + repl + content[m.end():]
    return content

## Step 3: HyDE (Hypothetical Document Embeddings)

One extra LLM call: write a hypothetical answer passage → embed →
fuse as a 3rd RRF arm (0.4/0.3/0.3). Fixes embedding mismatch for
paraphrased questions. Switch: `HYDE_ENABLED = False` → plain 0.5/0.5.

In [4]:
# =====================================================================
# HyDE — Hypothetical Document Embeddings (optional, 3rd RRF arm)
# =====================================================================
# For paraphrased questions the query words rarely match the document
# words (failure mode #2: embedding mismatch). HyDE: the LLM writes a
# short hypothetical answer passage; we embed it and fuse it as a third
# RRF arm. One extra LLM call, no reindexing.

HYDE_ENABLED = True          # switch: False -> plain 0.5/0.5 hybrid
W_VEC_HYDE, W_BM25_HYDE, W_HYDE = 0.4, 0.3, 0.3

def generate_hyde_passage(question: str) -> str:
    """One LLM call: a short passage as the answer would be phrased
    inside a post-production service offer. Thinking off, temp 0."""
    prompt = (
        "Schreibe einen kurzen, sachlichen Absatz (2-3 Saetze), wie die "
        "Antwort auf die folgende Frage in einem Post-Production-Leistungs-"
        "angebot formuliert waere. Nutze typische Fachbegriffe aus solchen "
        "Angeboten (z. B. Zahlungsziel, Skonto, Lieferzeit, Leistungsumfang, "
        "Nettobetrag). Antworte NUR mit dem Absatz, ohne Einleitung.\n\n"
        f"Frage: {question}"
    )
    return _strip_think(llm_chat(prompt)).strip()

def hybrid_search_hyde(query, top_n=RERANK_TOP_N, angebot_id=None):
    """Hybrid search with the optional HyDE arm (3rd RRF list)."""
    if not HYDE_ENABLED:
        return hybrid_search(query, top_n=top_n, angebot_id=angebot_id)
    passage = generate_hyde_passage(query)
    print(f"   hyde: {passage[:120]}...")
    return hybrid_search(query, top_n=top_n, angebot_id=angebot_id,
                         hyde_passage=passage)

print(f"✅ HyDE {'ON' if HYDE_ENABLED else 'OFF'} "
      f"(weights {W_VEC_HYDE}/{W_BM25_HYDE}/{W_HYDE} when on)")


✅ HyDE ON (weights 0.4/0.3/0.3 when on)


## Why Breadth Routes? (and where HyDE fits)

Cross-offer questions ("wie viele Angebote …", "vergleiche … über alle
Angebote") break the rerank paradigm: the reranker asks *"does THIS chunk
answer the question?"* — but breadth questions need **all** matching offers,
and top-k retrieval is capped by construction (with top-20 you can never
count more than 20). Similarity is not a completeness criterion.

**Statistics route** (count / mean / outliers): full scan — the LLM reads
every offer once (map → structured JSON), then **deterministic code**
reduces (count, percent, mean, outliers). No retrieval, no top-k cap, no LLM
in the reduce → exact numbers, complete coverage (every offer read).
Cost: one LLM call per offer — acceptable at this corpus size; at 5,000
documents this would switch to a vector pre-filter with score threshold +
verification (roadmap, Phase 2).

**Comparison route** (tree_summarize principle): topic retrieval (top-20,
dedup by offer) → LLM map: one fact line per offer → LLM reduce: the
comparison. Hand-rolled (≈25 lines) instead of LlamaIndex's built-in
`tree_summarize` so we keep our hybrid BM25+vector+RRF retrieval.

**HyDE** (Hypothetical Document Embeddings): for paraphrased questions the
query words rarely match the document words (failure mode #2 in the course
video). The LLM writes a short hypothetical answer passage; it is embedded
and fused as a **third RRF arm** (0.4/0.3/0.3 vs 0.5/0.5 without HyDE).
One extra LLM call, no reindexing.


## Step 4: Breadth Routes (code)

**Statistics:** full scan (map → JSON facts per offer) + deterministic code reduce.
**Comparison:** topic retrieval → LLM map (one line per offer) → LLM reduce.

In [5]:
# =====================================================================
# Breadth routes — statistics (full scan + code reduce), comparison
# (map-reduce over topic retrieval), year list (metadata scan)
# =====================================================================

STATISTICS_RE = re.compile(
    r"\b(wie viele|wieviel|durchschnitt|ausreißer|ausreisser|"
    r"anteil|prozent|median|häufig|hoeufig)\b", re.IGNORECASE)
COMPARISON_RE = re.compile(
    r"\b(vergleiche|vergleich|unterschiede|unterschied|unterscheiden|"
    r"unterschiedlich|nebeneinander)\b", re.IGNORECASE)
YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")
OFFER_WORD_RE = re.compile(r"\b(angebot|angebote)\b", re.IGNORECASE)

def is_statistics(q):
    """Count/mean/outlier question -> needs FULL coverage, not top-k."""
    return bool(STATISTICS_RE.search(q))

def is_comparison(q):
    """Cross-offer comparison -> map-reduce over topic retrieval."""
    return bool(COMPARISON_RE.search(q))

def is_year_question(q):
    """Year-list question ("Welche Angebote sind im Jahr 2024?").

    A year plus an offer word -> the answer is a complete list from the
    index metadata, not a top-k retrieval. Questions about a single
    aspect of a year ("Wie hoch war der Preis im Jahr 2024?") do NOT
    match — they stay on the aggregation/statistics routes.
    """
    return bool(YEAR_RE.search(q) and OFFER_WORD_RE.search(q))

# ---------- year list: deterministic metadata scan (no LLM, no retrieval) ----------

def year_route(question):
    """List every offer of a year from the index metadata.

    Deterministic breadth answer: the `datum` metadata of every offer in
    the index is scanned, so the list is complete by construction — no
    retrieval, no top-k cap, no LLM.
    """
    year = YEAR_RE.search(question).group(0)
    by_offer = _offer_texts()
    matches = []
    for offer_id, nodes in by_offer.items():
        datum = str(next((n.metadata.get("datum") for n in nodes
                          if n.metadata.get("datum")), ""))
        if datum.startswith(year):
            matches.append((offer_id, datum))
    matches.sort(key=lambda t: t[1], reverse=True)
    if not matches:
        return ("Breadth",
                f"Kein Angebot aus dem Jahr {year} ist im Index vorhanden "
                f"(vollständiger Scan über {len(by_offer)} Angebote).", [])
    lines = " · ".join(f"{oid} ({_format_date(d) or d})" for oid, d in matches)
    content = (f"Im Index sind **{len(matches)} Angebote** aus dem Jahr {year} "
               f"(vollständiger Scan über alle {len(by_offer)} Angebote, "
               f"kein Retrieval): {lines}.")
    return "Breadth", content, []

# ---------- statistics: full scan (map) + deterministic code (reduce) ----------

def _offer_texts():
    """All chunk texts grouped by offer (complete corpus, no retrieval)."""
    by_offer = {}
    for n in node_list:
        oid = n.metadata.get("angebot_id")
        if oid:
            by_offer.setdefault(oid, []).append(n)
    return by_offer

FACTS_PROMPT = (
    "Du liest ein Angebot eines Filmstudios. Extrahiere NUR Fakten, die "
    "explizit im Text stehen. Antworte NUR mit einem JSON-Objekt dieser Form:\n"
    '{"zahlungsziel_tage": int|null, "skonto_prozent": float|null, '
    '"lieferzeit": string|null, "garantie": string|null, "leistungen": [string]}\n'
    "Regeln:\n"
    "- zahlungsziel_tage: Tage bis zur Zahlung (z. B. 14). 'netto 30 Tage' -> 30. "
    "0 ist KEIN gültiger Wert. Wenn kein Zahlungsziel genannt wird, MUSS null "
    "stehen (nicht 0, nicht 30 als Standard).\n"
    "- skonto_prozent: Skontoprozent (z. B. 5.0), sonst null.\n"
    "- lieferzeit: kurze Wiedergabe der Lieferfrist, sonst null.\n"
    "- garantie: kurze Wiedergabe der Garantie, sonst null.\n"
    "- leistungen: Liste der genannten Leistungen (z. B. 'Color Grading').\n\n"
    "TEXT:\n{text}"
)

def _extract_offer_facts(text):
    """One LLM call per offer -> structured facts (the MAP step)."""
    prompt = FACTS_PROMPT.replace("{text}", text[:12000])
    for attempt in range(3):
        raw = llm_chat(prompt)
        m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except json.JSONDecodeError:
                pass
        if attempt < 2:
            time.sleep(0.5 * (attempt + 1))
    return None

# In-memory cache for the full-scan facts: the corpus is static while the
# index is, so one scan per notebook session — otherwise every statistics
# question would cost one LLM call per offer again.
_FACTS_CACHE = {}

def full_scan_facts():
    """MAP: the LLM reads EVERY offer once -> structured facts.

    Complete coverage by construction (no retrieval, no top-k cap).
    Cached per index; offers whose facts could not be extracted are skipped.
    """
    key = str(CHROMA_DIR)
    if key in _FACTS_CACHE:
        return _FACTS_CACHE[key]
    by_offer = _offer_texts()
    facts, failed = {}, []
    t0 = time.time()
    for i, (oid, nodes) in enumerate(sorted(by_offer.items()), 1):
        text = "\n\n".join(n.text for n in nodes)
        preis = next((n.metadata.get("preis") for n in nodes
                      if n.metadata.get("preis") is not None), None)
        datum = next((n.metadata.get("datum") for n in nodes
                      if n.metadata.get("datum")), None)
        f = _extract_offer_facts(text)
        if f is None:
            failed.append(oid)
            continue
        f["preis"] = preis
        f["datum"] = datum
        facts[oid] = f
        print(f"   [{i}/{len(by_offer)}] {oid}: "
              f"zz={f.get('zahlungsziel_tage')} skonto={f.get('skonto_prozent')}")
    print(f"   full scan: {len(facts)}/{len(by_offer)} offers in "
          f"{time.time() - t0:.0f}s" + (f" (failed: {failed})" if failed else ""))
    _FACTS_CACHE[key] = facts
    return facts

def reduce_statistics(facts):
    """REDUCE: deterministic CODE — count / mean / range / outliers.

    No LLM: statistics questions need exact numbers, and an LLM that
    'reads N partial answers' estimates instead of computing.
    """
    total = len(facts)
    lines = [f"Über alle **{total} Angebote** im Index gilt "
             f"(vollständiger Scan, kein Retrieval):"]

    zz = {o: f["zahlungsziel_tage"] for o, f in facts.items()
          if (f.get("zahlungsziel_tage") or 0) > 0}
    if zz:
        values = list(zz.values())
        mean = sum(values) / len(values)
        lines.append(f"- **Zahlungsziel:** {len(zz)} von {total} Angeboten "
                     f"legen ein Zahlungsziel fest — durchschnittlich "
                     f"{mean:.0f} Tage (Spanne {min(values)}–{max(values)} Tage).")
    else:
        lines.append("- **Zahlungsziel:** kein Angebot legt ein Zahlungsziel fest.")

    prices = {o: f["preis"] for o, f in facts.items() if f.get("preis") is not None}
    if prices:
        values = list(prices.values())
        mean = sum(values) / len(values)
        line = (f"- **Nettobetrag:** Durchschnitt **{format_price(mean)}** "
                f"(n={len(values)}), Spanne {format_price(min(values))} – "
                f"{format_price(max(values))}.")
        if len(values) >= 3:
            sd = (sum((v - mean) ** 2 for v in values) / len(values)) ** 0.5
            if sd > 0:
                outliers = {o: p for o, p in prices.items() if p > mean + 2 * sd}
                if outliers:
                    line += " Ausreißer (> Mittelwert + 2σ): " + ", ".join(
                        f"{o} ({format_price(p)})"
                        for o, p in sorted(outliers.items()))
        lines.append(line)

    lines.append("\n*Basis: vollständiger Scan aller Angebote "
                 "(kein Retrieval, keine Top-k-Grenze).*")
    return "\n".join(lines)

def statistics_route(question):
    """Statistics question -> full scan (map) + code reduce. No retrieval."""
    facts = full_scan_facts()
    if not facts:
        return ("Statistics", "Keine Fakten konnten extrahiert werden.", [])
    return ("Statistics", reduce_statistics(facts), [])

# ---------- comparison: map-reduce over topic retrieval ----------

def comparison_route(question, hyde_passage=None, top_offers=15):
    """MAP-REDUCE (tree_summarize principle) over topic retrieval.

    Retrieve on the topic (top-20, dedup by offer) -> MAP: LLM writes one
    fact line per offer -> REDUCE: LLM compares the lines. Hand-rolled
    instead of LlamaIndex's built-in tree_summarize so we keep our hybrid
    BM25+vector+RRF retrieval pipeline.
    """
    chunks = hybrid_search(question, top_n=20, hyde_passage=hyde_passage)
    by_offer = {}
    for n in chunks:
        oid = n.metadata.get("angebot_id")
        if oid:
            by_offer.setdefault(oid, []).append(n)
    offers = list(by_offer.items())[:top_offers]
    if not offers:
        return ("Comparison", "Keine passenden Angebote gefunden.", [])
    print(f"   map: {len(offers)} distinct offers from {len(chunks)} chunks")
    lines = []
    for oid, nodes in offers:
        text = "\n\n".join(n.text for n in nodes)[:2500]
        prompt = (f"Zusammenfassung in EINER Zeile: Was sagt dieses Angebot zu: "
                  f"{question}\nText: {text}\n"
                  f"Antworte NUR mit der Zeile, beginnend mit '{oid}: '")
        line = _strip_think(llm_chat(prompt)).strip()
        if line:
            lines.append(line)
    if not lines:
        return ("Comparison", "Keine Vergleichsdaten extrahiert.", [])
    prompt = (
        "Vergleiche die folgenden Zeilen zu den Angeboten und schreibe eine "
        "kurze Vergleichszusammenfassung auf Deutsch. Zitiere nach jeder "
        "Aussage die Angebots-ID ohne Klammern, z. B. AG1001. "
        "Nenne Gemeinsamkeiten und Unterschiede. Kein externes Wissen.\n\n"
        "ZEILEN:\n" + "\n".join(lines) + "\n\nVERGLEICH:"
    )
    text = _strip_think(llm_chat(prompt)).strip()
    ranked = [nodes[0] for _, nodes in offers]
    return ("Comparison", text, ranked)

print("✅ Breadth routes ready (statistics: full scan + code reduce; "
      "comparison: map-reduce; year: metadata scan)")

✅ Breadth routes ready (statistics: full scan + code reduce; comparison: map-reduce; year: metadata scan)


## Step 5: The Router

Ported 1:1 from the app's `query.py`, extended with the breadth routes.
Order matters: ID → statistics → comparison → year → ambiguous → aggregation → default.

In [6]:
# =====================================================================
# The router — ported 1:1 from the app's src/rag_system/query.py,
# extended with the breadth routes (statistics, comparison, year)
# =====================================================================

AMBIGUOUS_TOPIC_RE = re.compile(
    r"\b(preis|netto|brutto|nettobetrag|gesamtpreis|summe|zahlung|"
    r"zahlungsbedingung|skonto|f[aä]llig|faellig|datum|termin|"
    r"honorar|kosten|preisbasis)\b", re.IGNORECASE)
PRICE_TOPIC_RE = re.compile(
    r"\b(preis|netto|brutto|nettobetrag|bruttopreis|gesamtpreis|summe|"
    r"honorar|kosten)\b", re.IGNORECASE)
# Aggregation signals — structurally impossible with top-k vector search.
# "welche" only counts when it asks for offers ("welche angebote"); a bare
# "welche" ("welche Formate …") is a normal RAG question.
AGGREGATION_RE = re.compile(
    r"\b(welche\s+angebote|alle|wie viele|mehr als|mindestens|gr[oö]ßer als|"
    r"groesser als|teurer als|g[uü]nstiger als|im jahr|in dem jahr)\b",
    re.IGNORECASE)

def is_ambiguous(q):
    """Price/date/term question without offer reference or year."""
    return bool(AMBIGUOUS_TOPIC_RE.search(q)) and not YEAR_RE.search(q)

def is_price_question(q):
    return bool(PRICE_TOPIC_RE.search(q))

def is_aggregation(q):
    return bool(AGGREGATION_RE.search(q))

def is_compound(q):
    """Several parts (>=2 '?' or ' und ') -> lenient gate (keep 5, thr 4.0)."""
    return q.count("?") >= 2 or bool(re.search(r"\bund\b", q, re.IGNORECASE))

def format_price(value):
    """German number format: 1.251,03 EUR."""
    if value is None:
        return ""
    return f"{value:,.2f} \u20ac".replace(",", "\u00a0").replace(".", ",").replace("\u00a0", ".")

def price_answer(offer_id, chunks):
    """Deterministic price answer from chunk metadata (no LLM)."""
    for node in chunks:
        price = node.metadata.get("preis")
        if price is not None:
            datum = node.metadata.get("datum") or ""
            date_part = f" (Datum: {datum})" if datum else ""
            return (f"Der Nettobetrag von Angebot **{offer_id}** beträgt "
                    f"**{format_price(price)}**{date_part}.")
    return None

def _rerank_and_gate(question, chunks, compound=False):
    """Rerank + refusal gate. Compound: keep 5, threshold 4.0; else keep 3, 5.0."""
    keep = 5 if compound else KEEP
    threshold = 4.0 if compound else REFUSAL_THRESHOLD
    ranked = llm_rerank(question, chunks, keep=keep)
    top = ranked[0].metadata.get("rerank_score", 0.0) if ranked else 0.0
    if top < threshold:
        msg = ("Ich konnte keine zuverlässige Antwort in den vorliegenden "
               f"Angeboten finden (beste Übereinstimmung: {top:.0f}/10).")
        return ranked, msg
    return ranked, None

def _hyde_for(question):
    """HyDE passage for retrieval, or None when disabled/failed."""
    if not HYDE_ENABLED:
        return None
    passage = generate_hyde_passage(question)
    return passage or None

def _offer_count():
    """Distinct offers in the index (not chunks)."""
    return len({m.get("angebot_id") for m in collection.get(include=["metadatas"])["metadatas"] if m})

def _clarify(question):
    """Deduplicated offer candidates for clarification chips (no LLM)."""
    q_emb = embed_model.get_text_embedding(question)
    res = collection.query(query_embeddings=[q_emb], n_results=10,
                           include=["metadatas"])
    seen, cands = set(), []
    for i, _id in enumerate(res["ids"][0]):
        md = res["metadatas"][0][i] or {}
        oid = md.get("angebot_id") or _id
        if oid in seen:
            continue
        seen.add(oid)
        cands.append({"angebot_id": oid,
                      "datum": md.get("datum") or "\u2014",
                      "preis": md.get("preis")})
    if not cands:
        return None
    lines = " \u00b7 ".join(
        f"{c['angebot_id']} ({c['datum']}"
        + (f", {format_price(c['preis'])}" if c["preis"] else "") + ")"
        for c in cands)
    return (f"Insgesamt liegen {_offer_count()} Angebote vor \u2014 "
            f"meinst du eines dieser? {lines}"), cands

def run_query(question):
    """The full router. Returns (route, text, ranked_nodes)."""
    m = AG_RE.search(question)
    offer_id = m.group(0) if m else None

    # 1) ID-aware retrieval: filter to the referenced offer, NO fallback.
    if offer_id:
        chunks = hybrid_search(question, top_n=RERANK_TOP_N, angebot_id=offer_id,
                               hyde_passage=_hyde_for(question))
        if not chunks:
            return ("Refusal",
                    f"Angebot **{offer_id}** wurde nicht in den vorliegenden "
                    "Angeboten gefunden.", [])
        if is_price_question(question):
            a = price_answer(offer_id, chunks)
            if a:
                return ("RAG (deterministic)", a, chunks)
        ranked, refusal = _rerank_and_gate(question, chunks, is_compound(question))
        if refusal:
            return ("Refusal", refusal, ranked)
        text, _, _ = answer(question, top_n=RERANK_TOP_N, keep=5 if is_compound(question) else KEEP,
                            _pre_ranked=ranked)
        return ("RAG", text, ranked)

    # 2) Statistics (count/mean/outliers) -> FULL SCAN + code reduce.
    #    Checked BEFORE ambiguous/aggregation: "wie viele ... Zahlungsziel"
    #    would otherwise hit the clarification or RAG path.
    if is_statistics(question):
        return statistics_route(question)

    # 3) Comparison (vergleiche/unterschiede) -> map-reduce over topic retrieval.
    if is_comparison(question):
        return comparison_route(question, hyde_passage=_hyde_for(question))

    # 4) Year-list question ("Welche Angebote sind im Jahr 2024?") ->
    #    deterministic metadata scan — complete list, no LLM, no retrieval.
    if is_year_question(question):
        return year_route(question)

    # 5) Ambiguous price/date/term question -> clarification (no LLM call).
    if is_ambiguous(question):
        c = _clarify(question)
        if c:
            return ("Clarify", c[0], [])

    # 6) Aggregation (welche angebote/alle) -> RAG + explicit limitation note.
    if is_aggregation(question):
        chunks = hybrid_search_hyde(question, top_n=RERANK_TOP_N)
        ranked, refusal = _rerank_and_gate(question, chunks, is_compound(question))
        if refusal:
            return ("Refusal", refusal, ranked)
        text, _, _ = answer(question, top_n=RERANK_TOP_N,
                            keep=5 if is_compound(question) else KEEP, _pre_ranked=ranked)
        text += ("\n\n*Hinweis: Ich kann nur die hier geladenen Angebote "
                 "vergleichen \u2014 eine vollständige Auswertung über alle "
                 "Angebote folgt mit dem SQL-Pfad.*")
        return ("RAG", text, ranked)

    # 7) Default: grounded RAG (with optional HyDE arm).
    chunks = hybrid_search_hyde(question, top_n=RERANK_TOP_N)
    ranked, refusal = _rerank_and_gate(question, chunks, is_compound(question))
    if refusal:
        return ("Refusal", refusal, ranked)
    text, _, _ = answer(question, top_n=RERANK_TOP_N,
                        keep=5 if is_compound(question) else KEEP, _pre_ranked=ranked)
    return ("RAG", text, ranked)

## Step 6: Smoke Test (7 questions)

One question per router path: deterministic price, refusal, clarify,
year breadth, compound RAG, statistics (full scan), comparison (map-reduce).

In [11]:
# =====================================================================
# Smoke test — one question per router path
# =====================================================================

SMOKE = [
    # 1) ID + price -> deterministic price answer (no LLM)
    "Wie hoch war der Preis von AG1001?",
    # 2) ID not in index -> refusal
    "Was steht in AG9999?",
    # 3) ambiguous price question -> clarification
    "Wie hoch war der Preis?",
    # 4) year breadth route -> deterministic metadata scan (no LLM)
    "Welche Angebote sind im Jahr 2024?",
    # 5) compound RAG (2 parts, lenient gate)
    "Wie ist die Abnahme geregelt und welche Formate werden geliefert?",
    # 6) statistics -> full scan + code reduce
    "Wie viele Angebote haben ein Zahlungsziel von unter 30 Tagen?",
    # 7) comparison -> map-reduce
    "Vergleiche die Zahlungsbedingungen der letzten Angebote.",
]

for q in SMOKE:
    print("=" * 70)
    print(f"Q: {q}")
    route, text, ranked = run_query(q)
    print(f"Route: {route}")
    print(text)
    if ranked:
        print("Top:", ", ".join(
            f"{n.metadata.get('angebot_id')} ({n.metadata.get('rerank_score', 0):.0f}/10)"
            for n in ranked))
    print()

Q: Wie hoch war der Preis von AG1001?
Route: RAG (deterministic)
Der Nettobetrag von Angebot **AG1001** beträgt **7.380,00 €**.
Top: AG1001 (0/10)

Q: Was steht in AG9999?
Route: Refusal
Angebot **AG9999** wurde nicht in den vorliegenden Angeboten gefunden.

Q: Wie hoch war der Preis?
Route: Clarify
Insgesamt liegen 10 Angebote vor — meinst du eines dieser? AG1005 (—) · AG1001 (—, 7.380,00 €) · AG1003 (—, 19.300,00 €) · AG1006 (—, 8.300,00 €) · AG1004 (—, 2.320,00 €) · AG1002 (—, 5.650,00 €) · AG1007 (—, 11.500,00 €) · AG1008 (2024-09-16, 3.850,00 €) · AG1010 (2025-06-28, 6.950,00 €) · AG1009 (2025-02-05, 11.800,00 €)

Q: Welche Angebote sind im Jahr 2024?
Route: Breadth
Im Index sind **1 Angebote** aus dem Jahr 2024 (vollständiger Scan über alle 10 Angebote, kein Retrieval): AG1008 (16.09.2024).

Q: Wie ist die Abnahme geregelt und welche Formate werden geliefert?
   hyde: Der Leistungsumfang umfasst die Lieferung der finalen Masterdateien in den vereinbarten Formaten (z. B. ProRes 42